# Acoustic probes of WavLM Base, compared with the original tables (gate A6)

Run at a fixed commit on a Colab runtime with a CPU, after
`notebooks/02_migrate_wavlm_features.ipynb` and
`notebooks/05_silence_measurements.ipynb` have passed. No GPU is needed and
no encoder is loaded: the notebook reads the stored unpadded features of the
training set and the audio of the sampled files. Choose a high-RAM runtime:
the probes use every core of the runtime (`N_JOBS` is the number of cores),
and the notebook prints the number of cores and a timed estimate before the
long part.

5,000 training files are sampled (`probes.stratified_sample`). Their
acoustic targets (mean F0, mean HNR, spectral tilt; Praat through
Parselmouth, pinned) are computed on the clean audio and on the audio
trimmed at 40 dB; the silence and length targets come from the measurement
table of `notebooks/05_silence_measurements.ipynb`. A ridge probe is fitted
for every layer, target and scheme (random folds and speaker folds), with
the real target and with a permuted one. The chunks of the targets and the
layer cache stay on the local disk of the runtime. Drive receives the target
tables, the finished blocks of the probes, the R² tables, one figure and the
check tables.

## Gate A6

Gate A6 is not part of the pre-registered analysis plan. Its definition is
written in `docs/decisions.md` (Section 2), before any run.
The gate passes when every check below passes; each check stops the notebook
where it fails.

1. **Sample.** The 5,000 sampled files equal the original sample, file by
   file and in the same order, with the same labels and attacks
   (`probes.compare_sample`). The speaker ids are those of the official
   ASVspoof 2019 LA training protocol; the sample must have the number of
   speakers of the original tables (`probes.N_SPEAKERS`,
   `probes.speaker_check`).
2. **Targets.** The newly computed acoustic targets against the original
   target tables, clean and trimmed at 40 dB, on `audio_id`, with the
   tolerance `verify.TARGET_TOLERANCE` (`probes.compare_targets`). The table
   also says which targets are identical, that is, differ by exactly 0.0.
3. **Probe code on the original targets.** The probes fitted on the original
   target tables against the original R² tables, both conditions, every
   layer, both schemes, real and permuted (`probes.compare_r2`). The mean
   R², its standard deviation and the R² of the five folds may differ by
   `probes.R2_TOLERANCE`; the median alpha, the number of files and the
   number of speakers must be equal. The fits go through LAPACK and BLAS,
   whose last digits depend on the CPU and on the number of threads, and the
   machine that wrote the original tables is not recorded. The largest
   difference is printed, and whether it is exactly 0.0.
4. **Probe code on the new targets.** By rule, not by a list: a target whose
   new values are identical to the original target table in check 2 is the
   same computation as in check 3 and is not probed again; every target with
   any difference, however small, is probed again with the new values
   (`probes.rerun_table` decides and the notebook prints it). The resulting
   table is compared with the original R² tables as in check 3.
5. **The probe table of the manuscript.** Every cell of the table of
   selected layers (clean audio, speaker folds), built from the new R² table
   and from the original one, has the same text (`probes.selected_cells_check`).
6. **The printed numbers of Section 4.8**, from the in-text registry
   (`intext.probe_entries`, `intext.STATED_PROBES`): the feature dimension,
   the number of files and of speakers; the R² of the leading silence; the
   R² of the length in the window on clean and on trimmed audio. Each number
   is rounded once to the digits the manuscript prints, except the R² "at
   least ..." on trimmed audio, which states a lower bound and is cut off
   (`intext.lower_bound`). None is typed into this notebook.

In the trimmed condition the comparison covers the five targets of this
package. The four targets that only the original table has are listed in the
check table as not ported; they are not in the manuscript.

The probes are resumable: every finished block (one target, one scheme, one
condition) is written to Drive, and a run that starts again reads the
finished blocks instead of computing them (`probes.probe_blocks`). A block
is only read if it was computed from the same features, targets and speaker
ids.

Reported and not part of the gate: the sha256 of the original files, the
timed estimate, the table of the targets that were probed again, and the
figure.

The steps:

1. what this run compares; the original files and tables; the cores;
2. inputs: the training set on the local disk, the sample and its speakers;
3. the stored features of the sample;
4. the acoustic targets and their comparison with the original targets;
5. the timed check and the estimate;
6. the probes on the original targets (check 3);
7. the probes on the new targets (check 4) and the R² table;
8. the probe table and the printed numbers;
9. the figure;
10. gate A6.

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Google Drive is not mounted"

### Repository at a fixed commit

Set `COMMIT` to the commit of this repository that is being checked: the
full hash or at least its first 7 characters. The repository is public and
is cloned without a token. If the Colab secret `GITHUB_TOKEN` exists and
this notebook has access to it, the clone uses it (a repository that is not
public needs one); the token is not kept in the clone and is replaced by
`***` in everything git prints.

In [ ]:
import subprocess

REPOSITORY = "Nooralabd/silence-shortcut-wavlm-layers"
COMMIT = ""          # the commit to run: the full hash or at least its first 7 characters
WORKDIR = "/content/silence-shortcut-wavlm-layers"

assert len(COMMIT) >= 7 and set(COMMIT) <= set("0123456789abcdef"), \
    "set COMMIT to the commit that is being checked (full hash or at least its first 7 characters)"
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:    # the secret does not exist, or this notebook has no access to it
    token = ""       # a public repository is cloned without a token


def git(*arguments):
    """Run git and print what it prints; the token is never printed."""
    result = subprocess.run(["git", *arguments], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***")
    if output:
        print(output)
    assert result.returncode == 0, "git failed: see the lines above"
    return output


def clone(repository, folder):
    """Clone a repository, or fetch into an existing clone; a token, if there is one, is not kept in it."""
    address = f"https://{token}@github.com/{repository}.git" if token else f"https://github.com/{repository}.git"
    if not os.path.isdir(folder):
        git("clone", "-q", address, folder)
        git("-C", folder, "remote", "set-url", "origin", f"https://github.com/{repository}.git")
    else:
        git("-C", folder, "fetch", "-q", address, "+refs/heads/*:refs/remotes/origin/*")


clone(REPOSITORY, WORKDIR)
git("-C", WORKDIR, "checkout", "-q", COMMIT)
git("-C", WORKDIR, "rev-parse", "HEAD")
%cd {WORKDIR}

In [ ]:
# Installation. torch, torchaudio and torchcodec are the builds that extracted
# the original features: the pinned versions for CUDA 13.0. If anything pinned
# is missing or has another version, everything is installed in one pass and
# the cell always stops for a restart: a kernel never continues after pip has
# run in it (the runtime has imported numpy before this cell, and a module
# that is already loaded stays loaded when pip replaces it on disk).
# On the run after the restart nothing is installed: the cell checks the
# installed versions and the versions of the modules that the kernel loads.
import importlib
import importlib.metadata
import re
import sys

TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
TORCH_FAMILY = ("torch", "torchaudio", "torchcodec")          # installed from TORCH_INDEX
INSTALLED_ONCE = os.path.join(os.path.dirname(WORKDIR), "installed_once.txt")
OTHERS = os.path.join(os.path.dirname(WORKDIR), "requirements_without_torch.txt")
CONSTRAINT = os.path.join(os.path.dirname(WORKDIR), "torch_constraint.txt")

with open(os.path.join(WORKDIR, "requirements.txt"), encoding="utf-8") as fh:
    requirements = [line.strip() for line in fh if line.strip() and not line.lstrip().startswith("#")]
pins = dict(line.split("==") for line in requirements if "==" in line)
BUILDS = {name: pins[name] + "+" + TORCH_INDEX.rsplit("/", 1)[1] for name in TORCH_FAMILY}


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def not_as_pinned():
    """{package: (installed, wanted)} for everything that is missing or has another version."""
    found = {}
    for line in requirements:
        name = re.split(r"[=<>!~ ]", line, maxsplit=1)[0]
        have, want = installed(name), BUILDS.get(name, pins.get(name))
        same = have is not None and (want is None or (have if name in BUILDS else have.split("+")[0]) == want)
        if not same:
            found[name] = (have, want)
    if installed("ssl-silence") is None:
        found["ssl-silence"] = (None, "this repository")
    return found


different = not_as_pinned()
if different:
    assert not os.path.exists(INSTALLED_ONCE), (
        f"the libraries were installed once already and these are still not as pinned: {different}. "
        "Stop: do not restart again")
    imported = sorted(name for name in (*TORCH_FAMILY, "transformers", "ssl_silence") if name in sys.modules)
    assert not imported, (f"{imported} were imported before the installation: "
                          "Runtime > Restart session, then run the notebook again from the top")
    with open(OTHERS, "w", encoding="utf-8") as fh:
        for line in requirements:
            if re.split(r"[=<>!~ ]", line, maxsplit=1)[0] not in TORCH_FAMILY:
                print(line, file=fh)
    with open(CONSTRAINT, "w", encoding="utf-8") as fh:
        for name in TORCH_FAMILY:
            print(f"{name}=={BUILDS[name]}", file=fh)
    # The build is part of the pin (torch==2.11.0+cu130): for a pin without it pip
    # keeps another build of the same version that the runtime ships (+cpu).
    FAMILY_PINS = " ".join(f"{name}=={BUILDS[name]}" for name in TORCH_FAMILY)
    # torchvision of the runtime is built for its own torch and breaks imports
    # with another one; this package does not use it.
    !pip uninstall -q -y torchvision
    !pip install -q {FAMILY_PINS} --index-url {TORCH_INDEX}
    !pip install -q -r {OTHERS} -c {CONSTRAINT}
    !pip install -q --no-deps -e .
    importlib.invalidate_caches()
    # Before the restart: everything on disk is as pinned, so the next run installs nothing.
    different = not_as_pinned()
    assert not different, f"pip did not install these as pinned: {different}: stop"
    with open(INSTALLED_ONCE, "w", encoding="utf-8") as fh:
        print("the pinned libraries were installed in this runtime", file=fh)
    # pip has run in this kernel: it always stops here.
    raise RuntimeError("The pinned libraries were installed. "
                       "Runtime > Restart session, then run the notebook again from the top")

# Nothing was installed in this run. The modules that this kernel loads must
# be the pinned versions too, not only the files on disk.
LOADED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "scikit-learn": "sklearn",
          "torch": "torch", "torchaudio": "torchaudio", "transformers": "transformers"}
for name, module in LOADED.items():
    loaded = importlib.import_module(module).__version__
    assert loaded.split("+")[0] == pins[name], (
        f"{module} {loaded} is loaded in this kernel and the pin is {pins[name]}: it was imported before pip "
        "replaced it. Runtime > Restart session, then run the notebook again from the top")

import torch
import torchaudio
import transformers

print("loaded:", {name: sys.modules[module].__version__ for name, module in LOADED.items()})
print("torch", torch.__version__, "| CUDA of the build:", torch.version.cuda)
print("torchaudio", torchaudio.__version__, "| torchcodec", installed("torchcodec"),
      "| transformers", transformers.__version__)
if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))   # a running kernel does not see a new editable install

In [ ]:
import subprocess
import sys

import pandas as pd

from ssl_silence import config as cfg
from ssl_silence import provenance, verify

pd.set_option("display.max_rows", 200, "display.width", 200)
HEAD = provenance.git_commit()     # the full hash, with "-dirty" if a tracked file was changed
assert HEAD.startswith(COMMIT) and not HEAD.endswith("-dirty"), (HEAD, COMMIT)
print("commit:", HEAD)
print("root:", cfg.ROOT)

In [ ]:
# The quota of the Google account is read from the Drive API (this asks for
# access to the account); the mounted folder does not report it.
QUOTA = cfg.drive_quota()
print("Google Drive: {usage_gb:.1f} GB used of {limit_gb:.1f} GB, {free_gb:.1f} GB free".format(**QUOTA),
      "(GB = 10^9 bytes; required: %d GB free)" % cfg.MIN_FREE_GB)
cfg.require_free_space(quota=QUOTA)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### Mapping file

`legacy_map.yaml` names the original folders, files and labels. It is kept on
Drive only, at `<root>/migration/legacy_map.yaml`, and must have no empty
value (`verify.load_map` stops otherwise).

In [ ]:
MAP_PATH = os.path.join(cfg.migration_dir(), "legacy_map.yaml")
assert os.path.exists(MAP_PATH), f"upload legacy_map.yaml to {MAP_PATH}"
legacy = verify.load_map(MAP_PATH)
FORBIDDEN = legacy["forbidden"]
CHECKS = cfg.checks_dir()
checks = {}


def record(name, table, stop=True):
    """Write a check table to Drive, show it, and stop if a row fails."""
    path = verify.write_check(table, name, CHECKS, FORBIDDEN)
    checks[name] = table
    display(table)
    if stop and "ok" in table.columns:
        verify.require_ok(table, name)
    return path

### What this run compares

Nothing is set here. The original sample, the two original target tables and
the two original R² tables are read from their migrated copies, whose sha256
is recorded. The target chunks and the layer cache stay on the local disk of
the runtime.

In [ ]:
from ssl_silence import datasets, plots, probes, shards, silence
from ssl_silence.paper import intext
from ssl_silence.paper import tables as paper_tables

ENCODER = "wavlm-base"
SPLIT = "train"
CONDITIONS = probes.GATE_CONDITIONS
N_JOBS = os.cpu_count()                                   # every core of the runtime
ORIGINAL = os.path.join(cfg.migration_dir(), "original")
OUT = os.path.join(cfg.tables_dir(ENCODER), "probes")     # Drive: the target tables and the tables per condition
BLOCKS = os.path.join(OUT, "blocks")                      # Drive: the finished blocks of the probes
LOCAL = os.path.join(cfg.LOCAL_CACHE, "probes")           # local disk of the runtime: the target chunks
CACHE = {condition: os.path.join(cfg.LOCAL_CACHE, ENCODER, "unpadded", condition) for condition in CONDITIONS}
MEASUREMENTS = os.path.join(cfg.measurements_dir(), f"{SPLIT}.csv")
assert not cfg.on_drive(LOCAL), f"the target chunks would be written to Drive: {LOCAL}"

original_paths = [os.path.join(ORIGINAL, "files", probes.ORIGINAL_SAMPLE)] \
    + [os.path.join(ORIGINAL, "files", probes.ORIGINAL_TARGETS[condition]) for condition in CONDITIONS] \
    + [os.path.join(ORIGINAL, "tables", f"{probes.ORIGINAL_TABLES[condition]}.csv") for condition in CONDITIONS]
missing = [path for path in original_paths if not os.path.exists(path)]
assert not missing, f"run notebooks/02_migrate_wavlm_features.ipynb first: {missing} are missing"
assert os.path.exists(MEASUREMENTS), f"run notebooks/05_silence_measurements.ipynb first: {MEASUREMENTS} is missing"
record("probe_original_files", pd.DataFrame({
    "file": [os.path.relpath(path, ORIGINAL).replace(os.sep, "/") for path in original_paths],
    "sha256": [provenance.sha256_file(path) for path in original_paths]}), stop=False)
print("cores of the runtime:", N_JOBS, "| Praat:", probes.require_praat())
print("target chunks and layer cache (local disk):", cfg.LOCAL_CACHE, "| tables and blocks (Drive):", OUT)

### Inputs

The archive of ASVspoof 2019 LA is unpacked to the local disk of the runtime.
The speaker of every file is read from the official training protocol
(`datasets.items`). The sample and its speakers are the first two checks of
the gate.

In [ ]:
datasets.ensure_asvspoof(cfg.la_zip())
file_list = datasets.items(SPLIT)
print(SPLIT, datasets.check_items(file_list, SPLIT))
sample = probes.stratified_sample(file_list)
record("probe_sample", probes.compare_sample(
    sample, probes.read_targets(os.path.join(ORIGINAL, "files", probes.ORIGINAL_SAMPLE))))
record("probe_speakers", probes.speaker_check(sample))
groups = sample["speaker"].astype(str).to_numpy()
measurements = silence.silence_table(silence.read_measurements(MEASUREMENTS))

### The stored features of the sample

The training shards of the unpadded features, clean and trimmed at 40 dB, are
copied once into a layer cache on the local disk; the mean-pooled features of
the sampled files are read from it, in the order of the sample.

In [ ]:
features, layers = {}, {}
for condition in CONDITIONS:
    shards.build_layer_cache(cfg.feature_dir(ENCODER, "unpadded", condition), SPLIT, CACHE[condition])
    features[condition], layers[condition] = probes.load_sample_features(
        CACHE[condition], SPLIT, sample["audio_id"].astype(str).tolist())
    print(condition, "features:", features[condition].shape, "| layers:", layers[condition])

### The acoustic targets

Gate A6, check 2. The targets are computed in chunks on the local disk; the
finished table of each condition is written to Drive with the Praat settings
and versions. The table below the check says, for every target, whether it
is probed again with the new values (check 4).

In [ ]:
new_targets, original_targets, parts = {}, {}, []
for condition in CONDITIONS:
    new_targets[condition] = probes.compute_targets(sample, os.path.join(LOCAL, condition), condition)
    provenance.write_table(new_targets[condition], os.path.join(OUT, condition, probes.TARGETS_FILE),
                           condition=condition, praat_settings=probes.PRAAT, **probes.require_praat())
    original_targets[condition] = probes.in_sample_order(
        probes.read_targets(os.path.join(ORIGINAL, "files", probes.ORIGINAL_TARGETS[condition])), sample)
    parts.append(probes.compare_targets(new_targets[condition], original_targets[condition], condition))
target_check = pd.concat(parts, ignore_index=True)
record("probe_targets", target_check)
rerun = probes.rerun_table(target_check)
record("probe_rerun", rerun, stop=False)
again = {condition: rerun[(rerun["condition"] == condition) & rerun["probed_again"]]["target"].tolist()
         for condition in CONDITIONS}
print("probed again with the new targets (check 4):", again)

### Timed check

As many probes as the runtime has cores are fitted side by side in each
scheme, as the long part fits them, and the time is scaled to the number of
probes of this run. These probes are discarded. Blocks that a run before
this one has finished are not computed again, so a run that starts again
takes less than the estimate.

In [ ]:
n_probes = sum((len(probes.TARGETS[condition]) + len(again[condition])) * len(layers[condition]) * 2
               for condition in CONDITIONS)
first = CONDITIONS[0]
timing = probes.time_probes(features[first], new_targets[first], probes.ACOUSTIC_TARGETS[0], groups,
                            n_probes, N_JOBS)
record("probe_timing", timing, stop=False)
print("cores:", N_JOBS, "| probes per scheme:", n_probes,
      "| estimate of the probes:", round(float(timing["estimated_minutes"].sum()), 1), "minutes")

### The probes on the original targets

Gate A6, check 3. The original acoustic targets with the silence and length
targets of the measurement table; one block per target and scheme, written
to Drive when it is finished.

In [ ]:
silence_names = {condition: [name for name in probes.TARGETS[condition] if name in probes.SILENCE_TARGETS]
                 for condition in CONDITIONS}
r2_original, parts = {}, []
for condition in CONDITIONS:
    targets = probes.attach_silence_targets(original_targets[condition], measurements, condition,
                                            silence_names[condition])
    r2_original[condition] = probes.probe_blocks(
        features[condition], targets, probes.TARGETS[condition], groups, layers[condition], BLOCKS,
        f"{condition}__original_targets", n_jobs=N_JOBS)
    parts.append(probes.compare_r2(r2_original[condition], condition, legacy, ORIGINAL))
on_original = pd.concat(parts, ignore_index=True)
print("largest difference from the original tables:", probes.largest_difference(on_original),
      "| exactly 0.0:", probes.largest_difference(on_original) == 0.0)
record("probe_r2_original_targets", on_original)

### The probes on the new targets

Gate A6, check 4. Only the targets that differ from the original target
table are probed again. The R² table of both conditions is written to
`probe_r2.csv`, which the tables and the in-text numbers of the paper read.

In [ ]:
r2, parts = {}, []
for condition in CONDITIONS:
    redone = None
    if again[condition]:
        targets = probes.attach_silence_targets(new_targets[condition], measurements, condition,
                                                silence_names[condition])
        redone = probes.probe_blocks(features[condition], targets, again[condition], groups,
                                     layers[condition], BLOCKS, f"{condition}__new_targets", n_jobs=N_JOBS)
    r2[condition] = probes.merge_reruns(r2_original[condition], redone, probes.TARGETS[condition])
    provenance.write_table(probes.probe_table(r2[condition]), os.path.join(OUT, condition, "probe_r2_table.csv"))
    parts.append(probes.compare_r2(r2[condition], condition, legacy, ORIGINAL))
probe_r2 = probes.assemble(r2, ENCODER)
provenance.write_table(probe_r2, os.path.join(cfg.tables_dir(ENCODER), paper_tables.PROBE_R2),
                       n_sample=len(sample), n_folds=probes.N_FOLDS, cv_seed=probes.CV_SEED,
                       alphas=[float(a) for a in probes.ALPHAS], probed_again=again, **probes.praat_versions())
on_new = pd.concat(parts, ignore_index=True)
print("largest difference from the original tables:", probes.largest_difference(on_new),
      "| exactly 0.0:", probes.largest_difference(on_new) == 0.0)
record("probe_r2_new_targets", on_new)

### The probe table and the printed numbers

Gate A6, checks 5 and 6. The table of selected layers is written and shown.

In [ ]:
selected = probes.selected_layers_table(r2[paper_tables.BASE])
provenance.write_table(selected, os.path.join(OUT, paper_tables.BASE, "probe_r2_selected.csv"))
display(selected)
original_table = pd.read_csv(os.path.join(ORIGINAL, "tables", f"{probes.ORIGINAL_TABLES[paper_tables.BASE]}.csv"),
                             float_precision="round_trip")
record("probe_table_cells", probes.selected_cells_check(probe_r2, original_table))
stated = intext.compare_stated(intext.Tables.of({paper_tables.PROBE_R2: probe_r2}), intext.probe_entries(),
                               intext.STATED_PROBES)
record("probe_printed_numbers", stated)

### Figure

The probe R² per layer on clean audio. Not part of the gate; the first
author compares it with the published figure.

In [ ]:
figure = plots.probe_figure(r2[paper_tables.BASE], encoder=ENCODER)
print(plots.save_figure(figure, os.path.join(cfg.figures_dir(), "probe_r2"), print_width=False))

### Gate A6

Every check of the gate has passed where it was written; this cell shows
them together.

In [ ]:
gate = {name: checks[name] for name in probes.GATE_CHECKS}
display(verify.summarise_checks(gate))
for name, table in gate.items():
    verify.require_ok(table, name)
print("gate A6 passed | files sampled:", len(sample), "| speakers:", len(set(groups)),
      "| targets probed again:", sum(len(names) for names in again.values()),
      "| largest difference on the original targets:", probes.largest_difference(on_original),
      "| on the new targets:", probes.largest_difference(on_new),
      "| printed numbers reproduced:", len(stated))
print("tables written to", cfg.tables_dir(ENCODER), "| check tables written to", CHECKS)